In [1]:
from utils.data import ICLDataset
from utils.models import Model
from datasets import load_dataset
import os

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

# Dataset

In [3]:
pairs = (
    load_dataset("kh4dien/synonym-antonym", split="train")
    .filter(lambda x: x["type"] == "antonym")
    .map(lambda x: {"pairs": (x["input"], x["output"])})["pairs"]
)
pairs[:5]

[['flawed', 'perfect'],
 ['orthodox', 'unorthodox'],
 ['true', 'false'],
 ['daily', 'nightly'],
 ['distribution', 'concentration']]

In [4]:
ds = ICLDataset(pairs, bidirectional=True)

In [5]:
nshot_prompts, prompts, answers = ds.get_prompts(
    n_shot=5, n_shot_format="Q:{x}\nA:{y}\n", question_format="Q:{x}\nA:"
)

nshot_prompts[:2], prompts[:2], answers[:2]

399


(['Q:tragic\nA:comic\nQ:opaque\nA:transparent\nQ:fore\nA:aft\nQ:back\nA:face\nQ:real\nA:fictitious\nQ:dock\nA:',
  'Q:reversible\nA:irreversible\nQ:concentration\nA:distribution\nQ:criminal\nA:law-abiding citizen\nQ:similar\nA:contrary\nQ:universal\nA:specific\nQ:rest\nA:'],
 ['Q:dock\nA:', 'Q:rest\nA:'],
 ['ship', 'work'])

# Model

In [6]:
model = Model("meta-llama/Llama-2-7b-hf", layers_adr=["model", "layers"])
model.layers

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ModuleList(
  (0-31): 32 x LlamaDecoderLayer(
    (self_attn): LlamaAttention(
      (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
      (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    )
    (mlp): LlamaMLP(
      (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
      (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
      (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
      (act_fn): SiLUActivation()
    )
    (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
  )
)

# Experiments

In [7]:
responses = model.generate(nshot_prompts[:5], prompts[:5], layer=16, max_new_tokens=3, stops=["\n"])
responses, answers[:5]

You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


torch.Size([63])
tensor(0.0267, device='cuda:0', dtype=torch.bfloat16, grad_fn=<MeanBackward1>)


((['d', 'rest', 'real', 'The', 't'], ['d', 'rest', 'real', 'The', 't']),
 ['ship', 'work', 'virtual', 'attack', 'tough'])

In [8]:
responses_intervention = model.generate_with_intervention(
    prompts[:5],
    representations,
    layer=2,
    tokens_idx=[-1 for val in ds.get_token_indexes(prompts, token=":", tokenizer=model.tokenizer)],
    max_new_tokens=5,
    stops=[],
)
prompts[:5], responses_intervention[:5], answers[:5]

NameError: name 'representations' is not defined